# Stage 2 Fine-tuning: Explanatory Rewrite (Literal VI to Simplified VI)

This notebook contains the complete code to fine-tune the Stage 1 model to rewrite literal, complex machine translations into natural, beginner-friendly explanations. Run this notebook on **Kaggle GPU (T4 or P100)**.

In [ ]:
# 1. Install required libraries
!pip install -q transformers datasets accelerate huggingface_hub pandas evaluate sacrebleu

In [ ]:
# 2. Login to Hugging Face Hub
# If you attached HF_TOKEN in Kaggle Secrets, it will log in automatically.
import os
from huggingface_hub import login

hf_token = os.getenv("HF_TOKEN")
if hf_token:
    print("HF_TOKEN found in secrets. Logging in programmatically...")
    login(token=hf_token)
else:
    print("HF_TOKEN not found in environment. Prompting interactive login...")
    from huggingface_hub import notebook_login
    notebook_login()

In [ ]:
# 3. Load Stage 1 model and tokenizer as base
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Replace with your actual Stage 1 model ID on Hugging Face Hub
model_id = "your_hf_username/nllb-envi-ai-textbook-stage1"
print(f"Loading base model from: {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

# Force target language to Vietnamese during generation
model.config.forced_bos_token_id = tokenizer.convert_tokens_to_ids("vie_Latn")
print(f"Forced BOS Token ID set to Vietnamese: {model.config.forced_bos_token_id}")

In [ ]:
# 4. Load parallel training dataset (handmade_pairs.csv)
# Note: Upload 'handmade_pairs.csv' to Kaggle's working directory or dataset interface.
import pandas as pd
from datasets import Dataset

csv_file = "handmade_pairs.csv"
try:
    df = pd.read_csv(csv_file)
    print(f"Loaded {len(df)} sentence pairs for rewrite training.")
    dataset = Dataset.from_pandas(df)
except FileNotFoundError:
    print(f"Error: '{csv_file}' not found. Please upload the file to Kaggle.")

In [ ]:
# 5. Preprocess and tokenize data
# In Stage 2, both inputs and targets are in Vietnamese (vie_Latn)
max_input_length = 128
max_target_length = 128
src_lang = "vie_Latn"
tgt_lang = "vie_Latn"

def preprocess_function(examples):
    tokenizer.src_lang = src_lang
    tokenizer.tgt_lang = tgt_lang
    
    # Input is the literal machine translation, Target is the simplified rewrite
    inputs = [ex for ex in examples["vietnamese_literal"]]
    targets = [ex for ex in examples["vietnamese_rewrite"]]
    
    model_inputs = tokenizer(inputs, max_length=max_input_length, truncation=True)
    labels = tokenizer(text_target=targets, max_length=max_target_length, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Tokenize dataset
print("Preprocessing and tokenizing dataset...")
tokenized_dataset = dataset.map(preprocess_function, batched=True)

In [ ]:
# 6. Define evaluation metric (SacreBLEU)
import evaluate
import numpy as np

metric = evaluate.load("sacrebleu")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
        
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    
    # Replace -100 in the labels as we cannot decode them
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [[label.strip()] for label in decoded_labels]
    
    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {"bleu": result["score"]}

In [ ]:
# 7. Configure Seq2Seq training arguments
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir="nllb-envi-ai-textbook-stage2",
    evaluation_strategy="no", # Monitor training loss directly
    learning_rate=3e-5,
    per_device_train_batch_size=4,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=5, # Train for 5 epochs on the small rewrite dataset
    predict_with_generate=True,
    fp16=True,  # Mixed-precision training acceleration
    push_to_hub=True,
    hub_model_id="your_hf_username/nllb-envi-ai-textbook-stage2"  # Update with your HF username
)

In [ ]:
# 8. Setup Trainer and train
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Starting Stage 2 training...")
trainer.train()

In [ ]:
# 9. Push final rewrite model and tokenizer to Hugging Face Hub
trainer.push_to_hub()